# 03 — Dataset B Controlled Preprocessing Pilot

Thin Colab wrapper around the reusable `tdmec_pilot` modules. All logic lives in
`src/tdmec_pilot/` and `scripts/run_dataset_b_pilot.py` — this notebook only
mounts Google Drive, supplies configuration, and runs the pipeline.

**Safety:** processes exactly `statuses-2.xlsx` and `statuses-69.xlsx`; never the
other 68 files; no embeddings; no training; source files are read-only.

**Persistent outputs:** written under `<output_root>/pilot/<run_id>/` on the
mounted Drive. The pilot is resumable — re-run the last cell with the same
`--run-id` to continue after an interruption.

In [ ]:
# 1) (Colab only) Mount Google Drive as persistent storage.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/TDMEC_PROJECT_OUTPUTS'
except Exception:
    # Running on a normal Linux host / CI: use a local output root instead.
    OUTPUT_ROOT = './artifacts/pilot_outputs'
print('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
# 2) Get the code + dependencies. On Colab, clone the repo (or %cd into it).
import os, sys, subprocess
REPO_DIR = os.environ.get('TDMEC_REPO_DIR', '.')
if not os.path.exists(os.path.join(REPO_DIR, 'src', 'tdmec_pilot')):
    # Example clone (replace <REPO_URL>); skip if the repo is already present.
    # subprocess.run(['git', 'clone', '<REPO_URL>', 'community-evolution-modeling'], check=True)
    # REPO_DIR = 'community-evolution-modeling'
    pass
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                os.path.join(REPO_DIR, 'requirements.txt')], check=False)
print('src on path:', os.path.join(REPO_DIR, 'src'))

In [ ]:
# 3) Configuration. Provide the Dataset B source and the node-index map path.
#    DATASET_B_SOURCE: 'gdrive-anon:<folder_id>' | 'gdrive-api:<folder_id>' |
#                      'local:/path' | a mounted Drive folder path.
os.environ['DATASET_B_SOURCE'] = os.environ.get('DATASET_B_SOURCE', 'gdrive-anon:<dataset_b_folder_id>')
os.environ['PILOT_OUTPUT_ROOT'] = OUTPUT_ROOT
# Node-index map produced during discovery (16,736 authors -> 0..16,735).
# Build once with scripts/build_node_index_map.py, or point to the Drive copy:
os.environ['NODE_INDEX_MAP_PATH'] = os.environ.get(
    'NODE_INDEX_MAP_PATH', OUTPUT_ROOT + '/manifests/node_index_map.parquet')
CONFIG = os.path.join(REPO_DIR, 'configs', 'dataset_b_pilot.yaml')
print('config =', CONFIG)

In [ ]:
# 4) Run the pilot (thin call into the reusable pipeline).
from tdmec_pilot.config import load_pilot_config
from tdmec_pilot.pipeline import PilotPipeline

cfg = load_pilot_config(CONFIG)
pipe = PilotPipeline(
    cfg,
    dataset_b_source=os.environ['DATASET_B_SOURCE'],
    output_root=os.environ['PILOT_OUTPUT_ROOT'],
    node_index_map_path=os.environ['NODE_INDEX_MAP_PATH'],
    cache_root=os.environ.get('DISCOVERY_CACHE_ROOT', '/tmp/tdmec_cache'),
    # run_id='<existing_run_id>',  # <- set this to RESUME an interrupted run
)
report = pipe.run()
print('run_id     :', report['run_id'])
print('all_passed :', report['all_passed'])
print('accounting :', report['accounting'])
report['gates']

In [ ]:
# 5) Inspect the run outputs.
import json, pathlib
run_dir = pathlib.Path(pipe.layout.root)
for p in sorted(run_dir.rglob('*')):
    if p.is_file():
        print(p.relative_to(run_dir), p.stat().st_size, 'bytes')
print('\nvalidation_report.json:')
print(json.dumps(json.loads((run_dir / 'validation_report.json').read_text())['gates'], indent=2))

## Resume

If the run is interrupted, re-run cell 4 with `run_id='<the run_id printed
above>'`. Completed, checksum-verified chunks are skipped; only incomplete chunks
are reprocessed. A resume is refused if the canonical config hash differs
(`ConfigIncompatibleError`).

## CLI equivalent

```bash
python scripts/run_dataset_b_pilot.py \
  --config configs/dataset_b_pilot.yaml \
  --dataset-b-source "$DATASET_B_SOURCE" \
  --node-index-map "$NODE_INDEX_MAP_PATH" \
  --output-root "$PILOT_OUTPUT_ROOT"
```